In [0]:
from pyspark.sql.functions import col, lower, trim, when, current_timestamp, coalesce, to_date, lit

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print(" Silver schema ready")

In [0]:
from pyspark.sql.functions import expr

exchange_rates = (
    spark.table("azure_blob_storage.src_exchange_rates")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("currency", coalesce(lower(trim(col("currency"))), lit("none")))
    .withColumn("date", 
        expr("""
            coalesce(
                try_to_date(date, 'yyyy/MM/dd'),
                try_to_date(date, 'yyyy/M/d'),
                try_to_date(date, 'yyyy-MM-dd'),
                try_to_date(date, 'MM/dd/yyyy')
            )
        """)
    )
    .withColumn("rate_to_usd", when(col("rate_to_usd") <= 0, 1.0).otherwise(col("rate_to_usd")))
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["date", "currency"])
)

exchange_rates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.exchange_rates")
print(f" exchange_rates: {exchange_rates.count()} records")

In [0]:
print("\nSample data:")
display(spark.table("silver.exchange_rates").limit(10))